In [ ]:
!pip install tiktoken

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tiktoken

print("PyTorch version:", torch.__version__)
print("tiktoken loaded successfully!")

PyTorch version: 2.11.0+cpu
tiktoken loaded successfully!


In [ ]:
df = pd.read_csv('/content/IMDB Dataset (1).csv')

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
# Select 3,000 reviews randomly
df = df.sample(n=3000, random_state=42).reset_index(drop=True)

print("Dataset shape:", df.shape)


Dataset shape: (3000, 2)


In [ ]:
# Convert sentiment labels into numbers
# negative = 0
# positive = 1

df['label'] = df['sentiment'].map({
    'negative': 0,
    'positive': 1
})

print(df[['sentiment', 'label']].head())

  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0


In [ ]:
from sklearn.model_selection import train_test_split

# Reviews
X = df['review'].values

# Labels
y = df['label'].values

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 2400
Testing reviews: 600


In [ ]:
# Create the cl100k_base tokenizer
enc = tiktoken.get_encoding("cl100k_base")

print("Tokenizer:", enc.name)
print("Tokenizer loaded successfully!")

Tokenizer: cl100k_base
Tokenizer loaded successfully!


In [ ]:
# Tokenize the training and testing reviews

X_train_tokens = [enc.encode(review) for review in X_train]
X_test_tokens = [enc.encode(review) for review in X_test]

print("Number of training reviews:", len(X_train_tokens))
print("Number of testing reviews:", len(X_test_tokens))

# Show one example
print("\nFirst review:")
print(X_train[0])

print("\nFirst 20 token IDs:")
print(X_train_tokens[0][:20])

print("\nNumber of tokens in first review:", len(X_train_tokens[0]))

Number of training reviews: 2400
Number of testing reviews: 600

First review:
Well...I like this movie first of all because it's very well thought of... and well..because the um...director and others chose an extremely great actor to play Mike....and also my last reason because ( my opinion) Elijah Wood is so so hot!!!

First 20 token IDs:
[11649, 1131, 40, 1093, 420, 5818, 1176, 315, 682, 1606, 433, 596, 1633, 1664, 3463, 315, 1131, 323, 1664, 497]

Number of tokens in first review: 53


In [ ]:
MAX_LEN = 200

print("Maximum sequence length:", MAX_LEN)

Maximum sequence length: 200


In [ ]:
def pad_or_truncate(sequences, max_len):
    padded_sequences = []

    for seq in sequences:
        # Truncate if longer than max_len
        seq = seq[:max_len]

        # Pad with 0 if shorter than max_len
        if len(seq) < max_len:
            seq = seq + [0] * (max_len - len(seq))

        padded_sequences.append(seq)

    return padded_sequences


# Apply padding/truncation
X_train_padded = pad_or_truncate(X_train_tokens, MAX_LEN)
X_test_padded = pad_or_truncate(X_test_tokens, MAX_LEN)

print("Training reviews:", len(X_train_padded))
print("Testing reviews:", len(X_test_padded))

print("Length of first training sequence:", len(X_train_padded[0]))
print("Length of first testing sequence:", len(X_test_padded[0]))

Training reviews: 2400
Testing reviews: 600
Length of first training sequence: 200
Length of first testing sequence: 200


In [ ]:
# Find the largest token ID in the dataset
max_token_id = max(
    max(sequence)
    for sequence in X_train_padded + X_test_padded
)

# Vocabulary size
VOCAB_SIZE = max_token_id + 1

print("Maximum token ID:", max_token_id)
print("Vocabulary size:", VOCAB_SIZE)

Maximum token ID: 100252
Vocabulary size: 100253


In [ ]:
# Convert reviews into PyTorch tensors
X_train_tensor = torch.tensor(
    X_train_padded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_padded,
    dtype=torch.long
)

# Convert labels into PyTorch tensors
y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
)

print("Training input shape:", X_train_tensor.shape)
print("Testing input shape:", X_test_tensor.shape)
print("Training labels shape:", y_train_tensor.shape)
print("Testing labels shape:", y_test_tensor.shape)

print("\nInput data type:", X_train_tensor.dtype)
print("Label data type:", y_train_tensor.dtype)

Training input shape: torch.Size([2400, 200])
Testing input shape: torch.Size([600, 200])
Training labels shape: torch.Size([2400])
Testing labels shape: torch.Size([600])

Input data type: torch.int64
Label data type: torch.float32


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create datasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Batch size
BATCH_SIZE = 64

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 38
Testing batches: 10


In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentRNN, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # RNN layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(
            in_features=hidden_dim,
            out_features=1
        )

    def forward(self, x):
        # Convert token IDs to embeddings
        embedded = self.embedding(x)

        # Pass embeddings through RNN
        output, hidden = self.rnn(embedded)

        # Take the final hidden state
        hidden = hidden[-1]

        # Pass through linear layer
        output = self.fc(hidden)

        return output.squeeze(1)

In [ ]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

print(model)

SentimentRNN(
  (embedding): Embedding(100253, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the selected device
model = model.to(device)

# Loss function for binary classification
criterion = nn.BCEWithLogitsLoss()

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Using device:", device)
print("Loss function: BCEWithLogitsLoss")
print("Optimizer: Adam")

Using device: cpu
Loss function: BCEWithLogitsLoss
Optimizer: Adam


In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        # Move data to CPU/GPU
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Add loss
        total_loss += loss.item()

        # Calculate predictions
        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {average_loss:.4f} "
        f"Accuracy: {accuracy * 100:.2f}%"
    )

Epoch [1/5] Loss: 0.7022 Accuracy: 49.92%
Epoch [2/5] Loss: 0.6648 Accuracy: 59.04%
Epoch [3/5] Loss: 0.6329 Accuracy: 60.75%
Epoch [4/5] Loss: 0.5837 Accuracy: 66.54%
Epoch [5/5] Loss: 0.5399 Accuracy: 69.62%


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print("Correct predictions:", correct)
print("Total test reviews:", total)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Correct predictions: 301
Total test reviews: 600
Test Accuracy: 50.17%


In [ ]:
# Test the trained RNN with new reviews

new_reviews = [
    "This movie was amazing. I really enjoyed the story and acting.",
    "This was a terrible movie. It was boring and a waste of time.",
    "The movie was entertaining and the actors did a great job."
]

# Tokenize
new_tokens = [enc.encode(review) for review in new_reviews]

# Pad/truncate
new_padded = pad_or_truncate(new_tokens, MAX_LEN)

# Convert to PyTorch tensor
new_tensor = torch.tensor(
    new_padded,
    dtype=torch.long
).to(device)

# Make predictions
model.eval()

with torch.no_grad():
    outputs = model(new_tensor)
    probabilities = torch.sigmoid(outputs)

# Display results
for review, probability in zip(new_reviews, probabilities):

    probability = probability.item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Review:", review)
    print(f"Positive probability: {probability:.4f}")
    print("Prediction:", sentiment)
    print("-" * 60)

Review: This movie was amazing. I really enjoyed the story and acting.
Positive probability: 0.5181
Prediction: Positive
------------------------------------------------------------
Review: This was a terrible movie. It was boring and a waste of time.
Positive probability: 0.5656
Prediction: Positive
------------------------------------------------------------
Review: The movie was entertaining and the actors did a great job.
Positive probability: 0.3791
Prediction: Negative
------------------------------------------------------------


In [ ]:
print("Starting Activity 2: LSTM")

Starting Activity 2: LSTM


In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentLSTM, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(
            in_features=hidden_dim,
            out_features=1
        )

    def forward(self, x):
        # Convert token IDs into embeddings
        embedded = self.embedding(x)

        # Pass embeddings through LSTM
        output, (hidden, cell) = self.lstm(embedded)

        # Take the final hidden state
        hidden = hidden[-1]

        # Linear output
        output = self.fc(hidden)

        return output.squeeze(1)


print("LSTM model class created successfully!")

LSTM model class created successfully!


In [ ]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

lstm_model = SentimentLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

print(lstm_model)

SentimentLSTM(
  (embedding): Embedding(100253, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [ ]:
# Move the LSTM model to the same device
lstm_model = lstm_model.to(device)

# Loss function
criterion_lstm = nn.BCEWithLogitsLoss()

# Optimizer
optimizer_lstm = optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

print("LSTM is ready for training!")
print("Loss function: BCEWithLogitsLoss")
print("Optimizer: Adam")

LSTM is ready for training!
Loss function: BCEWithLogitsLoss
Optimizer: Adam


In [24]:
EPOCHS = 5

for epoch in range(EPOCHS):
    lstm_model.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer_lstm.zero_grad()

        # Forward pass
        outputs = lstm_model(inputs)

        # Calculate loss
        loss = criterion_lstm(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer_lstm.step()

        # Track loss
        total_loss += loss.item()

        # Calculate predictions
        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {average_loss:.4f} "
        f"Accuracy: {accuracy * 100:.2f}%"
    )

Epoch [1/5] Loss: 0.6936 Accuracy: 50.42%
Epoch [2/5] Loss: 0.6732 Accuracy: 59.25%
Epoch [3/5] Loss: 0.6341 Accuracy: 63.21%
Epoch [4/5] Loss: 0.5690 Accuracy: 66.79%
Epoch [5/5] Loss: 0.4872 Accuracy: 72.12%


In [25]:
lstm_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        # Get LSTM predictions
        outputs = lstm_model(inputs)

        # Convert output to 0 or 1
        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

lstm_test_accuracy = correct / total

print("Correct predictions:", correct)
print("Total test reviews:", total)
print(f"LSTM Test Accuracy: {lstm_test_accuracy * 100:.2f}%")

Correct predictions: 306
Total test reviews: 600
LSTM Test Accuracy: 51.00%


In [26]:
# Check that the review, tokens, and label are correctly connected

index = 0

print("Original review:")
print(X_train[index])

print("\nLabel:")
print(y_train[index])

print("\nFirst 20 token IDs:")
print(X_train_padded[index][:20])

print("\nNumber of non-padding tokens:")
print(sum(token != 0 for token in X_train_padded[index]))

Original review:
Well...I like this movie first of all because it's very well thought of... and well..because the um...director and others chose an extremely great actor to play Mike....and also my last reason because ( my opinion) Elijah Wood is so so hot!!!

Label:
1

First 20 token IDs:
[11649, 1131, 40, 1093, 420, 5818, 1176, 315, 682, 1606, 433, 596, 1633, 1664, 3463, 315, 1131, 323, 1664, 497]

Number of non-padding tokens:
53


In [27]:
class ImprovedSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(ImprovedSentimentLSTM, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        # Bidirectional LSTM produces 2 * hidden_dim
        self.fc = nn.Linear(
            in_features=hidden_dim * 2,
            out_features=1
        )

    def forward(self, x):
        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        # Forward final hidden state
        forward_hidden = hidden[-2]

        # Backward final hidden state
        backward_hidden = hidden[-1]

        # Combine both directions
        hidden = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output.squeeze(1)


print("Improved LSTM class created successfully!")

Improved LSTM class created successfully!


In [28]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

improved_lstm_model = ImprovedSentimentLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

improved_lstm_model = improved_lstm_model.to(device)

print(improved_lstm_model)

ImprovedSentimentLSTM(
  (embedding): Embedding(100253, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [29]:
criterion_improved = nn.BCEWithLogitsLoss()

optimizer_improved = optim.Adam(
    improved_lstm_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Loss function: BCEWithLogitsLoss")
print("Optimizer: Adam")
print("Improved LSTM is ready for training!")

Loss function: BCEWithLogitsLoss
Optimizer: Adam
Improved LSTM is ready for training!


In [30]:
EPOCHS = 8

for epoch in range(EPOCHS):
    improved_lstm_model.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer_improved.zero_grad()

        outputs = improved_lstm_model(inputs)

        loss = criterion_improved(outputs, labels)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            improved_lstm_model.parameters(),
            max_norm=1.0
        )

        optimizer_improved.step()

        total_loss += loss.item()

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    # Validation
    improved_lstm_model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = improved_lstm_model(inputs)

            predictions = (
                torch.sigmoid(outputs) >= 0.5
            ).float()

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy*100:.2f}% | "
        f"Validation Accuracy: {val_accuracy*100:.2f}%"
    )

Epoch [1/8] Loss: 0.6940 | Train Accuracy: 50.96% | Validation Accuracy: 50.17%
Epoch [2/8] Loss: 0.6622 | Train Accuracy: 62.83% | Validation Accuracy: 55.00%
Epoch [3/8] Loss: 0.6094 | Train Accuracy: 68.17% | Validation Accuracy: 55.67%
Epoch [4/8] Loss: 0.5190 | Train Accuracy: 75.83% | Validation Accuracy: 59.67%
Epoch [5/8] Loss: 0.3882 | Train Accuracy: 83.92% | Validation Accuracy: 58.00%
Epoch [6/8] Loss: 0.2654 | Train Accuracy: 91.17% | Validation Accuracy: 59.00%
Epoch [7/8] Loss: 0.1294 | Train Accuracy: 95.67% | Validation Accuracy: 59.67%
Epoch [8/8] Loss: 0.0627 | Train Accuracy: 98.46% | Validation Accuracy: 60.17%


In [31]:
# Save the current best LSTM model

best_lstm_state = improved_lstm_model.state_dict()

print("Best LSTM model saved.")
print("Best validation accuracy: 60.17%")

Best LSTM model saved.
Best validation accuracy: 60.17%


In [32]:
class RegularizedSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(RegularizedSentimentLSTM, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(
            in_features=hidden_dim * 2,
            out_features=1
        )

    def forward(self, x):
        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        hidden = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output.squeeze(1)


print("Regularized LSTM class created successfully!")

Regularized LSTM class created successfully!


In [33]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 96

regularized_lstm = RegularizedSentimentLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
)

regularized_lstm = regularized_lstm.to(device)

print(regularized_lstm)

RegularizedSentimentLSTM(
  (embedding): Embedding(100253, 128, padding_idx=0)
  (lstm): LSTM(128, 96, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)


In [34]:
criterion_regularized = nn.BCEWithLogitsLoss()

optimizer_regularized = optim.Adam(
    regularized_lstm.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Regularized LSTM is ready for training!")

Regularized LSTM is ready for training!


In [35]:
EPOCHS = 8

for epoch in range(EPOCHS):
    regularized_lstm.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer_regularized.zero_grad()

        outputs = regularized_lstm(inputs)

        loss = criterion_regularized(outputs, labels)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            regularized_lstm.parameters(),
            max_norm=1.0
        )

        optimizer_regularized.step()

        total_loss += loss.item()

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total

    # Check validation accuracy
    regularized_lstm.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = regularized_lstm(inputs)

            predictions = (
                torch.sigmoid(outputs) >= 0.5
            ).float()

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy*100:.2f}% | "
        f"Validation Accuracy: {val_accuracy*100:.2f}%"
    )

Epoch [1/8] Loss: 0.6953 | Train Accuracy: 50.12% | Validation Accuracy: 52.50%
Epoch [2/8] Loss: 0.6722 | Train Accuracy: 62.12% | Validation Accuracy: 54.67%
Epoch [3/8] Loss: 0.6396 | Train Accuracy: 66.29% | Validation Accuracy: 58.17%
Epoch [4/8] Loss: 0.5683 | Train Accuracy: 71.79% | Validation Accuracy: 57.67%
Epoch [5/8] Loss: 0.4861 | Train Accuracy: 76.83% | Validation Accuracy: 60.83%
Epoch [6/8] Loss: 0.3701 | Train Accuracy: 85.88% | Validation Accuracy: 60.83%
Epoch [7/8] Loss: 0.2379 | Train Accuracy: 90.96% | Validation Accuracy: 63.00%
Epoch [8/8] Loss: 0.1432 | Train Accuracy: 95.88% | Validation Accuracy: 63.33%


In [36]:
# Save the best regularized LSTM model

best_regularized_lstm_state = regularized_lstm.state_dict()

best_validation_accuracy = 63.33

print("Best model saved!")
print(f"Best validation accuracy: {best_validation_accuracy:.2f}%")

Best model saved!
Best validation accuracy: 63.33%


In [37]:
# Evaluate the regularized LSTM on the unseen test set

regularized_lstm.eval()

correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = regularized_lstm(inputs)

        predictions = (
            torch.sigmoid(outputs) >= 0.5
        ).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

final_test_accuracy = correct / total

print("Correct predictions:", correct)
print("Total test reviews:", total)
print(f"Final Test Accuracy: {final_test_accuracy * 100:.2f}%")

Correct predictions: 380
Total test reviews: 600
Final Test Accuracy: 63.33%


In [38]:
# Test the trained LSTM with new reviews

new_reviews = [
    "This movie was amazing. I really enjoyed the story and acting.",
    "This was a terrible movie. It was boring and a waste of time.",
    "The movie was entertaining and the actors did a great job."
]

# Tokenize the new reviews using cl100k_base
new_tokens = [enc.encode(review) for review in new_reviews]

# Pad or truncate
new_padded = pad_or_truncate(new_tokens, MAX_LEN)

# Convert to PyTorch tensor
new_tensor = torch.tensor(
    new_padded,
    dtype=torch.long
).to(device)

# Make predictions
regularized_lstm.eval()

with torch.no_grad():
    outputs = regularized_lstm(new_tensor)
    probabilities = torch.sigmoid(outputs)

# Display results
for review, probability in zip(new_reviews, probabilities):

    probability = probability.item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Review:", review)
    print(f"Positive probability: {probability:.4f}")
    print("Prediction:", sentiment)
    print("-" * 60)

Review: This movie was amazing. I really enjoyed the story and acting.
Positive probability: 0.2727
Prediction: Negative
------------------------------------------------------------
Review: This was a terrible movie. It was boring and a waste of time.
Positive probability: 0.0031
Prediction: Negative
------------------------------------------------------------
Review: The movie was entertaining and the actors did a great job.
Positive probability: 0.1269
Prediction: Negative
------------------------------------------------------------
